# Лабораторная работа №3 "Сверточные нейронные сети"

In [1]:
# https://colab.research.google.com/drive/1Ti6CXlauT1MXd5IlYvCELCTsGRin6bfp?usp=sharing
# https://www.tensorflow.org/datasets/catalog/rock_paper_scissors
# https://www.youtube.com/watch?v=PcNpbtM3Ucc&t=730s

from functools import cache
from itertools import islice
from typing import IO, Any, Callable, Mapping, Optional, Sequence, Sized, Tuple, cast
from zipfile import ZipFile

import torch.nn as nn
from matplotlib import pyplot as plt
from PIL import Image as image
from PIL.Image import Image
from torch import Tensor
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import VisionDataset


def RPSClassifier() -> nn.Sequential:
    return nn.Sequential()


class ZipImageDataset[TSample, TTarget](VisionDataset):
    _zip: ZipFile
    _loader: Callable[[IO[bytes]], Image]
    _classes: Sequence[str]
    _target_mapper: Mapping[str, int]
    _dataset: Sequence[Tuple[str, int]]

    def __init__(
        self,
        zip_filename: str,
        loader: Optional[Callable[[IO[bytes]], Image]] = None,
        transform: Optional[Callable[[Image], TSample]] = None,
        target_transform: Optional[Callable[[int], TTarget]] = None,
    ) -> None:
        super().__init__(
            zip_filename, transform=transform, target_transform=target_transform
        )
        self._zip = ZipFile(zip_filename, "r")
        self._loader = loader or ZipImageDataset.default_loader

        self._classes = self._build_classes()
        self._target_mapper = self._build_target_mapper()
        self._dataset = self._build_dataset()

    @property
    def classes(self) -> Sequence[str]:
        return self._classes

    def close(self) -> None:
        self._zip.close()

    def _build_classes(self) -> Sequence[str]:
        directories = [e for e in self._zip.infolist() if e.is_dir()]
        max_deep = max(e.filename.count("/") for e in directories)

        return sorted(
            ZipImageDataset._get_target(e.filename)
            for e in directories
            if e.filename.count("/") == max_deep
        )

    def _build_target_mapper(self) -> Mapping[str, int]:
        return {e: i for i, e in enumerate(self._classes)}

    def _build_dataset(self) -> Sequence[Tuple[str, int]]:
        return [
            (e.filename, self._target_mapper[ZipImageDataset._get_target(e.filename)])
            for e in self._zip.infolist()
            if not e.is_dir()
        ]

    def _unpack_file(self, filename: str) -> IO[bytes]:
        return self._zip.open(filename)

    def __len__(self) -> int:
        return len(self._dataset)

    def __getitem__(self, index: int) -> Tuple[TSample, TTarget]:
        path, target = self._dataset[index]

        with self._unpack_file(path) as stream:
            sample = self._loader(stream)

        if self.transform is not None:
            sample = self.transform(sample)

        if self.target_transform is not None:
            target = self.target_transform(target)

        return cast(TSample, sample), cast(TTarget, target)

    def __enter__(self) -> "ZipImageDataset":
        return self

    def __exit__(self, *_) -> None:
        self.close()

    @staticmethod
    def _get_target(path: str) -> str:
        return path.split("/")[-2]

    @staticmethod
    def default_loader(stream: IO[bytes]) -> Any:
        img = image.open(stream)
        return img.convert("RGB")


class CachedDataset[TSample, TTarget](Dataset):
    def __init__(self, dataset: Dataset) -> None:
        self._dataset = dataset

    @cache
    def __len__(self) -> int:
        if isinstance(self._dataset, Sized):
            return len(self._dataset)

        raise NotImplementedError()

    @cache
    def __getitem__(self, index: int) -> Tuple[TSample, TTarget]:
        return self._dataset[index]

    def __getattr__(self, attr: str) -> Any:
        return getattr(self._dataset, attr)


def show_samples(samples: Sequence[Tuple[Tensor, str]]) -> None:
    size = len(samples)
    plt.subplots(1, size, figsize=(12, 12))

    for i, (sample, target) in enumerate(samples):
        plt.subplot(1, size, i + 1)
        plt.title(target.title())

        # Normalize the image tensor from range [-1, 1] to [0, 1]
        sample = sample / 2 + 0.5

        # Convert the tensor to a NumPy array and transpose the dimensions from (channels, height, width) to (height, width, channels)
        image = sample.numpy().transpose(1, 2, 0).squeeze()
        plt.imshow(image)

    plt.tight_layout()
    plt.show()

## Загрузка данных

In [2]:
BATCH_SIZE = 64
transform = transforms.Compose(
    [
        transforms.Resize(150),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5) * 3, std=(0.5) * 3),
    ]
)

train = CachedDataset(ZipImageDataset("data/rps.zip", transform=transform))
train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)

test = CachedDataset(ZipImageDataset("data/rps-test-set.zip", transform=transform))
test_loader = DataLoader(test, batch_size=BATCH_SIZE, shuffle=False)

## Проверка предобработанных данных

In [ ]:
images, labels = next(iter(train_loader))
samples = ((s, train.classes[t]) for s, t in zip(images, labels))

show_samples([*islice(samples, 5)])